In [ ]:
# ================================================================
# GENERAL 2D STEADY HEAT CONDUCTION USING FDM
# ================================================================
#
# Governing equation:
#
#       k( d²T/dx² + d²T/dy² ) + S = 0
#
# or
#
#       d²T/dx² + d²T/dy² = -S/k
#
# Boundary conditions:
#
#       T(0,y)  = T_left
#       T(Lx,y) = T_right
#       T(x,0)  = T_bottom
#       T(x,Ly) = T_top
#
# Assumptions:
#   1. Steady state
#   2. Two-dimensional conduction
#   3. No bulk fluid motion
#   4. Constant thermal conductivity k
#   5. Uniform volumetric heat generation S
#
# ================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ================================================================
# 1. USER INPUTS
# ================================================================
#
# CHANGE THESE VALUES FOR DIFFERENT 2D PROBLEMS
# ================================================================

Lx = 1.0            # Plate length in x-direction (m)
Ly = 1.0            # Plate height in y-direction (m)

Nx = 10             # Total nodes in x-direction
Ny = 10             # Total nodes in y-direction

T_left = 0.0        # Boundary temperature at x = 0 (°C)
T_right = 0.0       # Boundary temperature at x = Lx (°C)
T_bottom = 0.0      # Boundary temperature at y = 0 (°C)
T_top = 100.0       # Boundary temperature at y = Ly (°C)

k = 1.0             # Thermal conductivity (W/m.K)

S = 0.0             # Volumetric heat generation (W/m³)


# ================================================================
# 2. SELECT LINEAR SOLVER
# ================================================================
#
# Options:
#
#       "gauss-seidel"
#
#       "direct"
#
# ================================================================

solver = "gauss-seidel"


# Gauss-Seidel settings

initial_guess = 0.0

tolerance = 1.0e-8

max_iterations = 10000

show_first_iterations = 5


# Fourier-series terms used for analytical comparison

analytical_terms = 100


# ================================================================
# 3. BASIC INPUT CHECKS
# ================================================================

if Nx < 3 or Ny < 3:
    raise ValueError("Nx and Ny must both be at least 3.")

if Lx <= 0 or Ly <= 0:
    raise ValueError("Lx and Ly must be positive.")

if k <= 0:
    raise ValueError("Thermal conductivity k must be positive.")


# ================================================================
# 4. COMPUTATIONAL GRID
# ================================================================

dx = Lx / (Nx - 1)

dy = Ly / (Ny - 1)


x = np.linspace(0, Lx, Nx)

y = np.linspace(0, Ly, Ny)


N_int_x = Nx - 2

N_int_y = Ny - 2

N_unknowns = N_int_x * N_int_y


# Grid-spacing ratio

beta2 = (dx / dy)**2


print("================================================")
print("GRID INFORMATION")
print("================================================")

print("Domain length Lx           =", Lx, "m")
print("Domain height Ly           =", Ly, "m")

print("Number of x-nodes Nx       =", Nx)
print("Number of y-nodes Ny       =", Ny)

print("Grid spacing dx            =", dx, "m")
print("Grid spacing dy            =", dy, "m")

print("Interior nodes in x        =", N_int_x)
print("Interior nodes in y        =", N_int_y)

print("Total unknown temperatures =", N_unknowns)

print("beta² = (dx/dy)²           =", beta2)

print()


# ================================================================
# 5. FINITE DIFFERENCE EQUATION
# ================================================================
#
# PDE:
#
#       T_xx + T_yy = -S/k
#
#
# Central difference approximation:
#
#       T_E - 2T_P + T_W
#       -----------------
#              dx²
#
#                 +
#
#       T_N - 2T_P + T_S
#       -----------------
#              dy²
#
#                 = -S/k
#
#
# Define:
#
#       beta² = dx²/dy²
#
#
# Positive-diagonal form:
#
#       2(1 + beta²)T_P
#
#       - T_W - T_E
#
#       - beta² T_S - beta² T_N
#
#       = S dx²/k
#
#
# If dx = dy:
#
#       beta² = 1
#
# therefore:
#
#       4T_P - T_W - T_E - T_S - T_N
#
#       = S dx²/k
#
#
# For Laplace equation S = 0:
#
#       4T_P - T_W - T_E - T_S - T_N = 0
#
# ================================================================


source_term = S * dx**2 / k


print("================================================")
print("FINITE DIFFERENCE EQUATION")
print("================================================")

print(
    "2(1+beta²)T_P - T_W - T_E "
    "- beta²T_S - beta²T_N = S dx²/k"
)

print()

print("Source contribution S dx²/k =", source_term)

print()


# ================================================================
# 6. INITIALIZE COMPLETE TEMPERATURE GRID
# ================================================================

T_fdm = np.full(
    (Ny, Nx),
    initial_guess,
    dtype=float
)


# ------------------------------------------------
# Apply boundary conditions
# ------------------------------------------------

T_fdm[0, :] = T_bottom

T_fdm[-1, :] = T_top

T_fdm[:, 0] = T_left

T_fdm[:, -1] = T_right


# ------------------------------------------------
# CORNER TREATMENT
# ------------------------------------------------
#
# If two different prescribed temperatures meet at a corner,
# the mathematical boundary data are discontinuous there.
#
# The isolated corner value does NOT enter any interior
# five-point stencil for this problem.
#
# For visualization only, we use the average of the two
# adjoining boundary temperatures.
# ------------------------------------------------

T_fdm[0, 0] = 0.5 * (T_bottom + T_left)

T_fdm[0, -1] = 0.5 * (T_bottom + T_right)

T_fdm[-1, 0] = 0.5 * (T_top + T_left)

T_fdm[-1, -1] = 0.5 * (T_top + T_right)


# ================================================================
# 7. 2D-TO-1D INDEX MAPPING
# ================================================================
#
# Interior-node ordering:
#
#       (1,1), (2,1), ...,
#       then next row in y.
#
# For Nx = Ny = 4:
#
#       T11, T21, T12, T22
#
# ================================================================


def get_row(i_int, j_int):

    return j_int * N_int_x + i_int


# ================================================================
# 8. BUILD MATRIX SYSTEM A T = b
# ================================================================


def build_matrix_system():

    A = np.zeros(
        (N_unknowns, N_unknowns)
    )

    b = np.full(
        N_unknowns,
        source_term
    )


    for j_int in range(N_int_y):

        for i_int in range(N_int_x):

            row = get_row(
                i_int,
                j_int
            )


            # ------------------------------
            # CENTRAL NODE
            # ------------------------------

            A[row, row] = (
                2.0 * (1.0 + beta2)
            )


            # ------------------------------
            # WEST
            # ------------------------------

            if i_int > 0:

                A[
                    row,
                    get_row(i_int - 1, j_int)
                ] = -1.0

            else:

                b[row] += T_left


            # ------------------------------
            # EAST
            # ------------------------------

            if i_int < N_int_x - 1:

                A[
                    row,
                    get_row(i_int + 1, j_int)
                ] = -1.0

            else:

                b[row] += T_right


            # ------------------------------
            # SOUTH
            # ------------------------------

            if j_int > 0:

                A[
                    row,
                    get_row(i_int, j_int - 1)
                ] = -beta2

            else:

                b[row] += (
                    beta2 * T_bottom
                )


            # ------------------------------
            # NORTH
            # ------------------------------

            if j_int < N_int_y - 1:

                A[
                    row,
                    get_row(i_int, j_int + 1)
                ] = -beta2

            else:

                b[row] += (
                    beta2 * T_top
                )


    return A, b


# ================================================================
# 9. DISPLAY MATRIX FOR SMALL CLASSROOM EXAMPLES
# ================================================================

if N_unknowns <= 16:

    A_small, b_small = build_matrix_system()


    print("================================================")
    print("COEFFICIENT MATRIX A")
    print("================================================")

    print(A_small)

    print()


    print("================================================")
    print("RIGHT-HAND SIDE VECTOR b")
    print("================================================")

    print(b_small)

    print()


# ================================================================
# 10. GAUSS-SEIDEL SOLVER
# ================================================================
#
# Rearranged FDM equation:
#
#
#            T_W + T_E
#
#          + beta²(T_S + T_N)
#
#          + S dx²/k
#
# T_P = ----------------------------
#
#             2(1 + beta²)
#
#
# Gauss-Seidel uses each newly calculated
# temperature immediately.
#
# ================================================================


def gauss_seidel_solver(T):

    denominator = (
        2.0 * (1.0 + beta2)
    )


    iteration_history = []


    # ------------------------------------------------
    # Record initial guess
    # ------------------------------------------------

    if N_unknowns <= 16:

        initial_values = []

        for j in range(1, Ny - 1):

            for i in range(1, Nx - 1):

                initial_values.append(
                    T[j, i]
                )

        iteration_history.append(
            [0] + initial_values
        )


    # ------------------------------------------------
    # Iteration loop
    # ------------------------------------------------

    for iteration in range(
        1,
        max_iterations + 1
    ):

        max_change = 0.0


        # Sweep:
        #
        # bottom --> top
        # left   --> right

        for j in range(1, Ny - 1):

            for i in range(1, Nx - 1):


                old_value = T[j, i]


                T_W = T[j, i - 1]

                T_E = T[j, i + 1]

                T_S = T[j - 1, i]

                T_N = T[j + 1, i]


                new_value = (

                    T_W
                    + T_E

                    + beta2 * (
                        T_S + T_N
                    )

                    + source_term

                ) / denominator


                # ------------------------------------
                # Immediate update:
                # defining feature of Gauss-Seidel
                # ------------------------------------

                T[j, i] = new_value


                change = abs(
                    new_value - old_value
                )

                max_change = max(
                    max_change,
                    change
                )


        # ------------------------------------------------
        # Save first few iterations
        # ------------------------------------------------

        if (
            N_unknowns <= 16
            and iteration <= show_first_iterations
        ):

            values = []

            for j in range(1, Ny - 1):

                for i in range(1, Nx - 1):

                    values.append(
                        T[j, i]
                    )

            iteration_history.append(
                [iteration] + values
            )


        # ------------------------------------------------
        # Convergence criterion
        # ------------------------------------------------

        if max_change < tolerance:

            return (
                T,
                iteration,
                max_change,
                iteration_history
            )


    return (
        T,
        max_iterations,
        max_change,
        iteration_history
    )


# ================================================================
# 11. DIRECT MATRIX SOLVER
# ================================================================


def direct_solver():

    A, b = build_matrix_system()


    # Important:
    #
    # FDM produced A and b.
    #
    # np.linalg.solve only solves
    #
    #           A T = b

    T_interior = np.linalg.solve(
        A,
        b
    )


    T = T_fdm.copy()


    for j_int in range(N_int_y):

        for i_int in range(N_int_x):

            row = get_row(
                i_int,
                j_int
            )

            T[
                j_int + 1,
                i_int + 1
            ] = T_interior[row]


    return T


# ================================================================
# 12. SOLVE
# ================================================================

if solver.lower() == "gauss-seidel":

    (
        T_fdm,
        iterations,
        final_change,
        history

    ) = gauss_seidel_solver(
        T_fdm
    )


    print("================================================")
    print("GAUSS-SEIDEL CONVERGENCE")
    print("================================================")

    print(
        "Iterations required =",
        iterations
    )

    print(
        "Final maximum change =",
        final_change
    )

    print(
        "Tolerance =",
        tolerance
    )

    print()


elif solver.lower() == "direct":

    T_fdm = direct_solver()

    history = None


    print("================================================")
    print("DIRECT MATRIX SOLUTION COMPLETED")
    print("================================================")

    print()


else:

    raise ValueError(
        "solver must be 'gauss-seidel' or 'direct'"
    )


# ================================================================
# 13. DISPLAY FIRST GAUSS-SEIDEL ITERATIONS
# ================================================================

if (
    solver.lower() == "gauss-seidel"
    and N_unknowns <= 16
):

    column_names = [
        "Iteration"
    ]


    for j in range(1, Ny - 1):

        for i in range(1, Nx - 1):

            column_names.append(
                f"T({i},{j})"
            )


    iteration_df = pd.DataFrame(
        history,
        columns=column_names
    )


    print("================================================")
    print("FIRST GAUSS-SEIDEL ITERATIONS")
    print("================================================")

    display(iteration_df)

    print()


# ================================================================
# 14. COMPLETE FDM TEMPERATURE GRID
# ================================================================

print("================================================")
print("FDM TEMPERATURE GRID")
print("================================================")

print()

print(
    "Rows correspond to increasing y."
)

print(
    "Columns correspond to increasing x."
)

print()

print(
    np.round(T_fdm, 5)
)

print()


# ================================================================
# 15. CHECK WHETHER ANALYTICAL SOLUTION IS AVAILABLE
# ================================================================
#
# The Fourier-series solution below applies only when:
#
#       S = 0
#
#       T_left   = 0
#
#       T_right  = 0
#
#       T_bottom = 0
#
#       T_top    = constant
#
# ================================================================


analytical_available = (

    np.isclose(S, 0.0)

    and np.isclose(
        T_left,
        0.0
    )

    and np.isclose(
        T_right,
        0.0
    )

    and np.isclose(
        T_bottom,
        0.0
    )
)


# ================================================================
# 16. FOURIER-SERIES ANALYTICAL SOLUTION
# ================================================================


def analytical_top_boundary(
    X,
    Y,
    terms=100
):

    T = np.zeros_like(
        X,
        dtype=float
    )


    for m in range(terms):


        # Odd numbers:
        #
        # n = 1, 3, 5, ...

        n = 2 * m + 1


        a = (
            n * np.pi / Lx
        )


        # ------------------------------------------------
        # Evaluate
        #
        # sinh(aY) / sinh(aLy)
        #
        # in numerically stable form.
        # ------------------------------------------------

        ratio = (

            np.exp(
                a * (Y - Ly)
            )

            * (
                1.0
                - np.exp(-2.0 * a * Y)
            )

            / (
                1.0
                - np.exp(-2.0 * a * Ly)
            )

        )


        T += (

            4.0 * T_top

            / (n * np.pi)

            * np.sin(
                n * np.pi * X / Lx
            )

            * ratio

        )


    return T


# ================================================================
# 17. ANALYTICAL VALUES AT FDM NODES
# ================================================================

X_mesh, Y_mesh = np.meshgrid(
    x,
    y
)


if analytical_available:


    # RAW Fourier-series values
    #
    # These are used for numerical comparison
    # at the INTERIOR nodes.

    T_exact_raw = analytical_top_boundary(
        X_mesh,
        Y_mesh,
        analytical_terms
    )


    print("================================================")
    print("ANALYTICAL SOLUTION AVAILABLE")
    print("================================================")

    print(
        "Fourier-series analytical solution "
        "is used as the benchmark."
    )

    print()


else:

    T_exact_raw = None


    print("================================================")
    print("ANALYTICAL COMPARISON NOT AVAILABLE")
    print("================================================")

    print(
        "The selected boundary conditions and/or "
        "source term do not satisfy the assumptions "
        "of this particular Fourier-series solution."
    )

    print()

    print(
        "The FDM solution itself is still valid."
    )

    print()


# ================================================================
# 18. INTERIOR-NODE COMPARISON TABLE
# ================================================================

nodes_data = []


for j in range(1, Ny - 1):

    for i in range(1, Nx - 1):


        row_data = {

            "Node (i,j)":
                f"({i},{j})",

            "x (m)":
                round(x[i], 5),

            "y (m)":
                round(y[j], 5),

            "T_FDM (°C)":
                round(
                    T_fdm[j, i],
                    6
                )

        }


        if analytical_available:

            exact_value = (
                T_exact_raw[j, i]
            )

            error = abs(
                T_fdm[j, i]
                - exact_value
            )


            row_data[
                "T_Exact (°C)"
            ] = round(
                exact_value,
                6
            )


            row_data[
                "Abs Error (°C)"
            ] = round(
                error,
                6
            )


            if abs(exact_value) > 1.0e-12:

                row_data[
                    "Relative Error (%)"
                ] = round(
                    100.0
                    * error
                    / abs(exact_value),
                    4
                )

            else:

                row_data[
                    "Relative Error (%)"
                ] = np.nan


        nodes_data.append(
            row_data
        )


results_df = pd.DataFrame(
    nodes_data
)


print("================================================")
print("INTERIOR-NODE RESULTS")
print("================================================")

display(results_df)


# ================================================================
# 19. VISUALIZATION
# ================================================================
#
# IMPORTANT:
#
# The FDM calculation is performed only at the grid nodes.
#
# contourf interpolates between those discrete nodes only
# for visualization.
#
# ================================================================


if analytical_available:


    # ------------------------------------------------------------
    # Dense analytical grid
    # ------------------------------------------------------------

    x_dense = np.linspace(
        0,
        Lx,
        200
    )

    y_dense = np.linspace(
        0,
        Ly,
        200
    )


    X_dense, Y_dense = np.meshgrid(
        x_dense,
        y_dense
    )


    # ------------------------------------------------------------
    # Raw truncated Fourier series
    # ------------------------------------------------------------

    T_exact_dense_raw = analytical_top_boundary(
        X_dense,
        Y_dense,
        analytical_terms
    )


    # ------------------------------------------------------------
    # COMMON PHYSICAL TEMPERATURE SCALE
    # ------------------------------------------------------------
    #
    # For this Laplace problem with S = 0,
    # the solution must remain between the
    # minimum and maximum boundary temperatures.
    #
    # ------------------------------------------------------------

    T_plot_min = min(
        T_left,
        T_right,
        T_bottom,
        T_top
    )

    T_plot_max = max(
        T_left,
        T_right,
        T_bottom,
        T_top
    )


    if np.isclose(
        T_plot_min,
        T_plot_max
    ):

        T_plot_max = (
            T_plot_min + 1.0
        )


    contour_levels = np.linspace(
        T_plot_min,
        T_plot_max,
        21
    )


    # ------------------------------------------------------------
    # VISUALIZATION-ONLY CORRECTION OF FOURIER RINGING
    # ------------------------------------------------------------
    #
    # A finite Fourier-series truncation may overshoot
    # very close to the discontinuous upper corners
    # (Gibbs phenomenon).
    #
    # The raw values above are NOT changed for the
    # interior-node numerical comparison.
    #
    # Here clipping is used ONLY so that the graphical
    # temperature map respects the physical range.
    # ------------------------------------------------------------

    T_exact_dense_plot = np.clip(
        T_exact_dense_raw,
        T_plot_min,
        T_plot_max
    )


    # ------------------------------------------------------------
    # Explicitly impose boundary temperatures on
    # analytical visualization
    # ------------------------------------------------------------

    T_exact_dense_plot[
        0,
        :
    ] = T_bottom


    T_exact_dense_plot[
        -1,
        1:-1
    ] = T_top


    T_exact_dense_plot[
        :,
        0
    ] = T_left


    T_exact_dense_plot[
        :,
        -1
    ] = T_right


    # Corner values for visualization only

    T_exact_dense_plot[
        0,
        0
    ] = 0.5 * (
        T_bottom + T_left
    )


    T_exact_dense_plot[
        0,
        -1
    ] = 0.5 * (
        T_bottom + T_right
    )


    T_exact_dense_plot[
        -1,
        0
    ] = 0.5 * (
        T_top + T_left
    )


    T_exact_dense_plot[
        -1,
        -1
    ] = 0.5 * (
        T_top + T_right
    )


    # ============================================================
    # THREE-PANEL FIGURE
    # ============================================================

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(19, 5.5)
    )


    # ============================================================
    # PANEL 1: FDM TEMPERATURE CONTOUR
    # ============================================================

    contour1 = axes[0].contourf(
        X_mesh,
        Y_mesh,
        T_fdm,
        levels=contour_levels,
        extend="neither"
    )


    fig.colorbar(
        contour1,
        ax=axes[0],
        label="Temperature (°C)"
    )


    # Actual computational nodes

    axes[0].scatter(
        X_mesh,
        Y_mesh,
        s=28
    )


    # ------------------------------------------------------------
    # Show nodal values for small classroom grids
    # ------------------------------------------------------------

    if Nx <= 8 and Ny <= 8:

        for j in range(Ny):

            for i in range(Nx):

                axes[0].annotate(

                    f"{T_fdm[j, i]:.1f}",

                    (
                        x[i],
                        y[j]
                    ),

                    xytext=(4, 4),

                    textcoords="offset points",

                    fontsize=8
                )


    axes[0].set_title(
        "FDM Temperature Contour\n"
        f"Nx = {Nx}, Ny = {Ny}"
    )

    axes[0].set_xlabel(
        "x (m)"
    )

    axes[0].set_ylabel(
        "y (m)"
    )

    axes[0].set_xlim(
        0,
        Lx
    )

    axes[0].set_ylim(
        0,
        Ly
    )


    # Show actual grid for small problems

    if Nx <= 10 and Ny <= 10:

        axes[0].set_xticks(x)

        axes[0].set_yticks(y)

        axes[0].grid(
            True,
            alpha=0.25
        )


    # ============================================================
    # PANEL 2: ANALYTICAL TEMPERATURE CONTOUR
    # ============================================================

    contour2 = axes[1].contourf(
        X_dense,
        Y_dense,
        T_exact_dense_plot,
        levels=contour_levels,
        extend="neither"
    )


    fig.colorbar(
        contour2,
        ax=axes[1],
        label="Temperature (°C)"
    )


    axes[1].set_title(
        "Analytical Temperature Contour\n"
        "(Fourier Series)"
    )

    axes[1].set_xlabel(
        "x (m)"
    )

    axes[1].set_ylabel(
        "y (m)"
    )

    axes[1].set_xlim(
        0,
        Lx
    )

    axes[1].set_ylim(
        0,
        Ly
    )


    # ============================================================
    # PANEL 3: ABSOLUTE ERROR AT FDM INTERIOR NODES
    # ============================================================
    #
    # The error is shown as DISCRETE POINTS because
    # the FDM solution exists at the grid nodes.
    #
    # ============================================================

    error_x = []

    error_y = []

    error_values = []


    for j in range(1, Ny - 1):

        for i in range(1, Nx - 1):

            error_x.append(
                x[i]
            )

            error_y.append(
                y[j]
            )

            error_values.append(

                abs(
                    T_fdm[j, i]
                    - T_exact_raw[j, i]
                )

            )


    error_values = np.array(
        error_values
    )


    error_scatter = axes[2].scatter(
        error_x,
        error_y,
        c=error_values,
        s=140
    )


    fig.colorbar(
        error_scatter,
        ax=axes[2],
        label="Absolute Error (°C)"
    )


    # Add numerical error values

    for xx, yy, ee in zip(
        error_x,
        error_y,
        error_values
    ):

        axes[2].annotate(

            f"{ee:.4f}",

            (xx, yy),

            xytext=(5, 5),

            textcoords="offset points",

            fontsize=9
        )


    axes[2].set_title(
        "FDM Discretization Error\n"
        "at Interior Nodes"
    )

    axes[2].set_xlabel(
        "x (m)"
    )

    axes[2].set_ylabel(
        "y (m)"
    )

    axes[2].set_xlim(
        0,
        Lx
    )

    axes[2].set_ylim(
        0,
        Ly
    )


    if Nx <= 10 and Ny <= 10:

        axes[2].set_xticks(x)

        axes[2].set_yticks(y)

        axes[2].grid(
            True,
            alpha=0.25
        )


    plt.tight_layout()

    plt.show()


else:

    # ============================================================
    # ONLY NUMERICAL FDM PLOT
    # when analytical solution is unavailable
    # ============================================================

    plt.figure(
        figsize=(8, 6)
    )


    contour = plt.contourf(
        X_mesh,
        Y_mesh,
        T_fdm,
        levels=20
    )


    plt.colorbar(
        contour,
        label="Temperature (°C)"
    )


    plt.scatter(
        X_mesh,
        Y_mesh,
        s=28
    )


    # Nodal values for small classroom grid

    if Nx <= 8 and Ny <= 8:

        for j in range(Ny):

            for i in range(Nx):

                plt.annotate(

                    f"{T_fdm[j, i]:.1f}",

                    (
                        x[i],
                        y[j]
                    ),

                    xytext=(4, 4),

                    textcoords="offset points",

                    fontsize=8
                )


    plt.xlabel(
        "x (m)"
    )

    plt.ylabel(
        "y (m)"
    )


    plt.title(
        "2D FDM Temperature Contour\n"
        f"Nx = {Nx}, Ny = {Ny}, S = {S}"
    )


    if Nx <= 10 and Ny <= 10:

        plt.xticks(x)

        plt.yticks(y)

        plt.grid(
            True,
            alpha=0.25
        )


    plt.tight_layout()

    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. 3D SURFACE VISUALIZATION
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the surface
surf = ax.plot_surface(X_mesh, Y_mesh, T_fdm, cmap='magma', edgecolor='none', alpha=0.8)

# Add labels and styling
ax.set_title('3D Temperature Profile: Steady State Heat Conduction', fontsize=14)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_zlabel('Temperature (°C)')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='T (°C)')
ax.view_init(elev=30, azim=45)
plt.show()

# 2. GRID CONVERGENCE ANALYSIS (CONCEPTUAL DEMO)
# We will calculate the Root Mean Square Error (RMSE) against the analytical solution
if analytical_available:
    fdm_interior = T_fdm[1:-1, 1:-1]
    exact_interior = T_exact_raw[1:-1, 1:-1]
    rmse = np.sqrt(np.mean((fdm_interior - exact_interior)**2))

    print('\n' + '='*48)
    print('ENHANCED NUMERICAL ANALYSIS')
    print('='*48)
    print(f'Root Mean Square Error (RMSE): {rmse:.6f} °C')
    print(f'Grid Density: {Nx}x{Ny} ({N_unknowns} degrees of freedom)')
    print('-'*48)
    print('Insight: Increasing Nx and Ny will reduce this error at O(dx²),')
    print('demonstrating the second-order accuracy of the central difference scheme.')

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

# We will run a fresh solver sweep and store the state at specific intervals
T_anim = np.full((Ny, Nx), initial_guess)
T_anim[0, :] = T_bottom
T_anim[-1, :] = T_top
T_anim[:, 0] = T_left
T_anim[:, -1] = T_right

frames = []
denominator = 2.0 * (1.0 + beta2)

# Record initial state
frames.append(T_anim.copy())

# Run a quick solver loop for animation
for k in range(1, 201):  # Capture first 200 iterations
    for j in range(1, Ny - 1):
        for i in range(1, Nx - 1):
            T_anim[j, i] = (T_anim[j, i-1] + T_anim[j, i+1] + beta2*(T_anim[j-1, i] + T_anim[j+1, i]) + source_term) / denominator

    if k % 5 == 0:  # Save every 5th iteration for smoothness
        frames.append(T_anim.copy())

# Setup the figure for animation
fig_anim, ax_anim = plt.subplots(figsize=(6, 5))
cont = ax_anim.contourf(X_mesh, Y_mesh, frames[0], levels=20, cmap='magma')
fig_anim.colorbar(cont, label='Temperature (°C)')
ax_anim.set_title("Iteration: 0")

def update(frame_idx):
    ax_anim.clear()
    c = ax_anim.contourf(X_mesh, Y_mesh, frames[frame_idx], levels=20, cmap='magma')
    ax_anim.set_title(f"Gauss-Seidel Convergence - Iteration: {frame_idx * 5}")
    ax_anim.set_xlabel("x (m)")
    ax_anim.set_ylabel("y (m)")
    return c

ani = animation.FuncAnimation(fig_anim, update, frames=len(frames), interval=50, blit=False)
plt.close() # Prevents static plot from showing

print("Generating convergence animation...")
HTML(ani.to_jshtml())

In [ ]:
# ================================================================
# 20. PARAMETRIC SENSITIVITY STUDY: EFFECT OF HEAT GENERATION (S)
# ================================================================

S_values = [0, 500, 1000, 2000, 5000] # W/m³
max_temps = []

print("Running parametric study...")

for S_val in S_values:
    # Define local source term
    local_source = S_val * dx**2 / k

    # Solve using the Direct Method for speed during iteration
    A_p, b_p = build_matrix_system()
    # Update b with the new source term
    b_p = np.full(N_unknowns, local_source)

    # Re-apply boundary conditions to the b vector
    # (Simplified for this study assuming standard boundaries from cell 1)
    A_sys, b_sys = build_matrix_system()
    # We just need to adjust b_sys for the new S
    # b_sys = (original boundary contributions) + S_new
    # Since build_matrix_system uses the global 'source_term', let's temporarily swap it
    source_term_backup = source_term
    import __main__
    __main__.source_term = local_source
    A_p, b_p = build_matrix_system()
    __main__.source_term = source_term_backup

    T_sol = np.linalg.solve(A_p, b_p)
    max_temps.append(np.max(T_sol))

# Visualization
plt.figure(figsize=(10, 6))
plt.plot(S_values, max_temps, 'o-', linewidth=2, markersize=8, color='firebrick')
plt.grid(True, linestyle='--', alpha=0.7)
plt.title("Parametric Study: Max Internal Temperature vs. Heat Generation", fontsize=14)
plt.xlabel("Volumetric Heat Generation S (W/m³)", fontsize=12)
plt.ylabel("Maximum Temperature (°C)", fontsize=12)

# Highlight the 'Passive' vs 'Active' cooling regimes
plt.axhspan(0, 100, alpha=0.1, color='green', label='Boundary Dominated')
plt.axhspan(100, max(max_temps), alpha=0.1, color='red', label='Generation Dominated')

plt.legend()
plt.show()

print(f"Study complete. Maximum observed temperature at S={S_values[-1]}: {max_temps[-1]:.2f}°C")

In [ ]:
# ================================================================
# 21. HEAT FLUX VECTOR FIELD VISUALIZATION (STREAMLINES)
# ================================================================

# Calculate gradients: q = -k * grad(T)
# np.gradient returns [dY, dX]
Ty, Tx = np.gradient(T_fdm, dy, dx)
qx = -k * Tx
qy = -k * Ty

plt.figure(figsize=(10, 8))

# Background temperature contour
cp = plt.contourf(X_mesh, Y_mesh, T_fdm, levels=20, cmap='magma', alpha=0.3)
plt.colorbar(cp, label='Temperature (°C)')

# Heat flux streamlines
# color by magnitude of flux
flux_magnitude = np.sqrt(qx**2 + qy**2)
stream = plt.streamplot(x, y, qx, qy, color=flux_magnitude, cmap='viridis',
                        linewidth=1.5, density=1.2, arrowsize=1.5)

plt.colorbar(stream.lines, label='Heat Flux Magnitude (W/m²)')

plt.title("Heat Flux Vector Field & Streamlines", fontsize=14)
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.grid(True, alpha=0.2)
plt.show()

print("Visualization complete: The arrows show the path of steepest temperature descent.")

In [ ]:
# ================================================================
# 22. SPECTRAL ANALYSIS: FOURIER HARMONIC DECOMPOSITION
# ================================================================
import numpy as np
import matplotlib.pyplot as plt

# Extract temperature profile along a horizontal line at y = 0.75*Ly
j_idx = int(0.75 * Ny)
y_pos = y[j_idx]
T_slice = T_fdm[j_idx, :]

# Perform Fast Fourier Transform (FFT) to see harmonic contributions
# We remove the DC offset (mean) to focus on the shape
coeffs = np.abs(np.fft.rfft(T_slice - np.mean(T_slice)))
freqs = np.fft.rfftfreq(Nx, d=dx)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot the spatial slice
ax1.plot(x, T_slice, 'o-', color='darkblue', label=f'T at y={y_pos:.2f}m')
ax1.set_title("Spatial Temperature Slice")
ax1.set_xlabel("x (m)")
ax1.set_ylabel("Temperature (°C)")
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot the frequency spectrum
ax2.stem(freqs, coeffs, basefmt=" ")
ax2.set_title("Fourier Spectral Magnitude (Harmonics)")
ax2.set_xlabel("Spatial Frequency (cycles/m)")
ax2.set_ylabel("Magnitude")
ax2.grid(True, alpha=0.3)

plt.suptitle("Spectral Analysis of the Thermal Field", fontsize=16)
plt.show()

print(f"Dominant spatial frequency detected at: {freqs[np.argmax(coeffs[1:])+1]:.2f} cycles/m")